# Carnatify — GPU feature extraction for the July data batch

Extracts Demucs+pyin raga features for the ~364 newly downloaded archive.org tracks
(`data/pending_extraction.json` in the repo — no Drive audio upload needed; each mp3 is
fetched straight from archive.org, processed, and deleted).

- Output: one npz per track in `Drive/carnatify_raga_cache_v3/` (same format as `data/raga_v2_cache/archive_v3/`).
- **Resumable** — rerun any time; already-extracted tracks are skipped.
- Last cell zips the new npz files to `Drive/carnatify_new_npz.zip` → download it and hand it back.

Runtime: GPU (T4 fine). Expect ~1.5–3 h for the full list.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install demucs soundfile librosa essentia
import essentia.standard as es
assert hasattr(es, 'TonicIndianArtMusic'), 'essentia broken'
print('essentia OK')
!rm -rf /content/carnatify
!git clone -q https://github.com/ShyamRavidath/carnatify.git /content/carnatify
import json
pending = json.load(open('/content/carnatify/data/pending_extraction.json'))
print(len(pending), 'tracks in pending list')

In [ ]:
import json, sys, time, traceback
from pathlib import Path
import requests

sys.path.insert(0, '/content/carnatify')
from raga_v2_pipeline import process_track

CACHE_DIR = Path('/content/drive/MyDrive/carnatify_raga_cache_v3')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/tmp_audio'); TMP.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({'User-Agent': (
    'CarnatifyResearchBot/1.0 (+https://carnatify.vercel.app; '
    'educational raga-classification research; contact: dpti0904@gmail.com)')})

pending = json.load(open('/content/carnatify/data/pending_extraction.json'))
done = {p.stem for p in CACHE_DIR.glob('*.npz')}
todo = [r for r in pending if r['track_id'] not in done]
print(f'{len(todo)} to extract ({len(pending)-len(todo)} already in Drive cache)')

n_ok = n_skip = n_fail = 0
t0 = time.time()
for k, rec in enumerate(todo, 1):
    mp3 = TMP / 'current.mp3'
    try:
        r = session.get(rec['url'], timeout=300)
        r.raise_for_status()
        mp3.write_bytes(r.content)
        res = process_track(rec['track_id'], rec['raga'], str(mp3), CACHE_DIR)
        if res is None:
            n_skip += 1; status = 'skip (short/empty)'
        else:
            n_ok += 1; status = f"ok ({res.get('tonic_method','?')})"
    except Exception as e:
        n_fail += 1; status = f'FAIL {e}'
        traceback.print_exc(limit=1)
    finally:
        mp3.unlink(missing_ok=True)
    el = time.time() - t0
    print(f'[{k}/{len(todo)}] {rec["track_id"][:70]} {status} '
          f'({el/60:.0f} min, ok={n_ok} skip={n_skip} fail={n_fail})', flush=True)
    time.sleep(1.0)
print('DONE', n_ok, 'ok,', n_skip, 'skipped,', n_fail, 'failed')

In [ ]:
# Zip every npz in the Drive cache that matches the pending list (i.e. this batch)
import json, zipfile
from pathlib import Path
CACHE_DIR = Path('/content/drive/MyDrive/carnatify_raga_cache_v3')
pending = json.load(open('/content/carnatify/data/pending_extraction.json'))
ids = {r['track_id'] for r in pending}
out = Path('/content/drive/MyDrive/carnatify_new_npz.zip')
n = 0
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in CACHE_DIR.glob('*.npz'):
        if p.stem in ids:
            z.write(p, p.name); n += 1
print(f'{n} npz -> {out}')
print('Download this zip and give it back (or share the Drive file).')